In [ ]:
# ============================================================
# 🎬 Movie & Book Review Sentiment + Rating Predictor
# Fine-tuning Llama-3.2-3B with QLoRA on Yelp Reviews
# ============================================================
# HOW TO USE:
#   1. In Google Colab: Runtime > Change runtime type > T4 GPU
#   2. Add HF_TOKEN to Colab Secrets (key icon on the left)
#   3. Replace YOUR_HF_USERNAME below with your HuggingFace username
#   4. Run cells top to bottom
# ============================================================
# ── STEP 1: Install dependencies ────────────────────────────
# Colab already has torch, transformers, bitsandbytes, datasets, accelerate
# We only need to install trl and peft
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "trl", "peft",
    "bitsandbytes>=0.46.1",
    "transformers>=4.50.0",
], check=True)
print("Dependencies installed")


In [ ]:
# ── STEP 2: Imports ─────────────────────────────────────────
import re
from tqdm import tqdm
from datetime import datetime

import numpy as np
import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from google.colab import userdata
from huggingface_hub import login




In [ ]:
# ── STEP 3: GPU Check (must pass before anything else) ──────

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\n  No GPU detected!\n"
        "  Fix: Runtime > Change runtime type > T4 GPU > Save\n"
        "  Then: Runtime > Disconnect and delete runtime\n"
        "  Then reconnect and re-run all cells.\n"
    )

capability = torch.cuda.get_device_capability()
use_bf16   = False  # force float16 for stable quantized training on T4
dtype      = torch.float16

print(f"Transformers : {transformers.__version__}")
print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"Capability   : {capability}  ->  using {'bfloat16' if use_bf16 else 'float16'}")




In [ ]:
# ── STEP 4: Constants & Hyperparameters ─────────────────────

# !! Replace this with your HuggingFace username !!
HF_USER = "YOUR_USERNAME"

# Identity
BASE_MODEL   = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "review"

# Dataset
# Yelp Review Full: ratings 1-5, 650k train / 50k test, free on HuggingFace
DATASET_NAME = "Yelp/yelp_review_full"
TRAIN_SAMPLE = 10_000   # set to 5_000 for a quick smoke-test (~20 min on T4)
TEST_SAMPLE  = 2000

# Run naming (auto-generated so each run has a unique HF repo)
RUN_NAME         = datetime.now().strftime('%Y-%m-%d_%H.%M.%S')
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME   = f"{HF_USER}/{PROJECT_RUN_NAME}"

# QLoRA
QUANT_4_BIT    = True
LORA_RANK      = 64
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.1
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# Training
LEARNING_RATE = 1e-4
BATCH_SIZE    = 4
GRAD_ACCUM    = 4        # effective batch size = 16
EPOCHS        = 1
MAX_SEQ_LEN   = 256
WARMUP_RATIO  = 0.03
LR_SCHEDULER  = "cosine"

print(f"\nModel will be saved to: https://huggingface.co/{HUB_MODEL_NAME}")




In [ ]:
# ── STEP 5: Log in to HuggingFace ───────────────────────────

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)
print("Logged in to HuggingFace")




In [ ]:
# ── STEP 6: Load & Inspect the Dataset ──────────────────────

raw = load_dataset(DATASET_NAME)
print(raw)
print("\nSample row:", raw['train'][0])




In [ ]:
# ── STEP 7: Build Prompt-Completion Pairs ───────────────────
# Yelp labels are 0-indexed:
#   0 = 1 star  -> Negative
#   1 = 2 stars -> Negative
#   2 = 3 stars -> Mixed
#   3 = 4 stars -> Positive
#   4 = 5 stars -> Positive

STAR_TO_SENTIMENT = {
    0: "Negative",
    1: "Negative",
    2: "Mixed",
    3: "Positive",
    4: "Positive",
}

def make_prompt(text: str) -> str:
    """Format a review into the model input prompt."""
    text = text[:800].strip()
    return (
        "Analyze the following review.\n"
        "Respond with ONLY: Sentiment: <Positive|Negative|Mixed>. Rating: <1-5> stars.\n\n"
        f"Review: {text}\n\n"
        "Prediction:"
    )

def make_completion(label: int) -> str:
    """Format the ground-truth label into the target completion."""
    return f" Sentiment: {STAR_TO_SENTIMENT[label]}. Rating: {label + 1} stars."

def format_row(row: dict) -> dict:
    """Combine prompt + completion into a single 'text' field for SFT."""
    return {"text": make_prompt(row["text"]) + make_completion(row["label"])}

# Sample, shuffle, and format
train_ds = raw['train'].shuffle(seed=42).select(range(TRAIN_SAMPLE))
test_ds  = raw['test'].shuffle(seed=42).select(range(TEST_SAMPLE))

train_formatted = train_ds.map(format_row)
test_formatted  = test_ds.map(format_row)

print(f"Train : {len(train_formatted):,} rows")
print(f"Test  : {len(test_formatted):,} rows")
print("\nExample formatted row:\n")
print(train_formatted[0]['text'])




In [ ]:
# ── STEP 8: Load Tokenizer + Quantized Base Model ───────────

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
) if QUANT_4_BIT else BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=dtype,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model = prepare_model_for_kbit_training(base_model)

print(f"Base model memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")




In [ ]:
# ── STEP 9: Apply LoRA Adapters ─────────────────────────────

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()




In [ ]:
# ── STEP 10: Train with SFTTrainer ──────────────────────────

training_args = SFTConfig(
    output_dir=f"/content/{PROJECT_RUN_NAME}",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    fp16=False,
    bf16=False,
    logging_steps=50,
    save_strategy="epoch",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_strategy="every_save",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_formatted,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()
print("Training complete!")
print(f"Model pushed to: https://huggingface.co/{HUB_MODEL_NAME}")


# ============================================================
# THE MOMENT OF TRUTH — Evaluate the Fine-Tuned Model
# ============================================================

def parse_prediction(text: str) -> tuple:
    """
    Extract (sentiment, rating) from raw model output.
    Returns (None, None) if the model did not follow the expected format.
    """
    rating_match = re.search(r'Rating:\s*(\d)', text, re.IGNORECASE)
    rating       = int(rating_match.group(1)) if rating_match else None

    sentiment = None
    for label in ["Positive", "Negative", "Mixed"]:
        if label.lower() in text.lower():
            sentiment = label
            break

    return sentiment, rating


def run_inference(model, tokenizer, prompt: str, max_new_tokens: int = 20) -> str:
    """Run greedy inference; return only the text after 'Prediction:'."""
    device = next(model.parameters()).device
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("Prediction:")[-1].strip() if "Prediction:" in decoded else decoded.strip()


def compute_metrics(results: list, model_prefix: str) -> tuple:
    """Return (MAE, sentiment_accuracy, parse_rate) for a given model."""
    rating_key    = f"{model_prefix}_rating"
    sentiment_key = f"{model_prefix}_sentiment"

    valid_rating    = [r for r in results if r[rating_key]    is not None]
    valid_sentiment = [r for r in results if r[sentiment_key] is not None]

    mae        = np.mean([abs(r['true_stars'] - r[rating_key]) for r in valid_rating])       if valid_rating    else float('nan')
    acc        = np.mean([r['true_sentiment'] == r[sentiment_key] for r in valid_sentiment]) if valid_sentiment else float('nan')
    parse_rate = len(valid_rating) / len(results)

    return mae, acc, parse_rate


# ── Load fine-tuned model ────────────────────────────────────
# We reload a clean base model before attaching the saved LoRA adapters.
# This avoids double-wrapping if training was run in the same session.
# In a fresh session set: HUB_MODEL_NAME = "your-username/review-YYYY-MM-DD_..."

eval_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
fine_tuned_model = PeftModel.from_pretrained(eval_base, HUB_MODEL_NAME)
fine_tuned_model.eval()
print(f"Fine-tuned model memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")


# ── Quick single-example sanity check ───────────────────────

sample         = test_ds[0]
true_stars     = sample['label'] + 1
true_sentiment = STAR_TO_SENTIMENT[sample['label']]
prompt         = make_prompt(sample['text'])

raw_output                  = run_inference(fine_tuned_model, tokenizer, prompt)
pred_sentiment, pred_rating = parse_prediction(raw_output)

print("=" * 60)
print("REVIEW (truncated):")
print(sample['text'][:300])
print("=" * 60)
print(f"TRUE  -> Sentiment: {true_sentiment:8s}  |  Rating: {true_stars} stars")
print(f"PRED  -> Sentiment: {str(pred_sentiment):8s}  |  Rating: {str(pred_rating)} stars")
print(f"Raw output: '{raw_output}'")


# ── Full evaluation loop ─────────────────────────────────────

EVAL_SIZE = 500   # raise to 500+ for more reliable numbers

results = []
for row in tqdm(test_ds.select(range(EVAL_SIZE)), desc="Evaluating"):
    true_label = row['label']
    prompt     = make_prompt(row['text'])

    ft_raw                  = run_inference(fine_tuned_model, tokenizer, prompt)
    ft_sentiment, ft_rating = parse_prediction(ft_raw)

    base_raw                    = run_inference(base_model, tokenizer, prompt)
    base_sentiment, base_rating = parse_prediction(base_raw)

    results.append({
        "true_stars"     : true_label + 1,
        "true_sentiment" : STAR_TO_SENTIMENT[true_label],
        "ft_rating"      : ft_rating,
        "ft_sentiment"   : ft_sentiment,
        "base_rating"    : base_rating,
        "base_sentiment" : base_sentiment,
    })

ft_mae,   ft_acc,   ft_parse   = compute_metrics(results, "ft")
base_mae, base_acc, base_parse = compute_metrics(results, "base")

print("\n" + "=" * 55)
print(f"{'':30s} {'Base':>10s}  {'Fine-tuned':>10s}")
print("-" * 55)
print(f"{'Rating MAE (lower = better)':30s} {base_mae:>10.3f}  {ft_mae:>10.3f}")
print(f"{'Sentiment Accuracy':30s} {base_acc:>10.2%}  {ft_acc:>10.2%}")
print(f"{'Parse Rate':30s} {base_parse:>10.2%}  {ft_parse:>10.2%}")
print("=" * 55)
print(f"\nRating MAE improvement  : {base_mae - ft_mae:+.3f} stars")
print(f"Sentiment accuracy gain : {ft_acc - base_acc:+.2%}")


# ── Live demo — paste any review here ───────────────────────

my_review = """
The pasta was absolutely phenomenal - perfectly cooked, rich sauce,
generous portions. The service was a bit slow but the staff were
genuinely friendly. I'd definitely come back for the food alone.
"""

prompt            = make_prompt(my_review.strip())
output            = run_inference(fine_tuned_model, tokenizer, prompt)
sentiment, rating = parse_prediction(output)

print("\nREVIEW:")
print(my_review.strip())
print("\nPREDICTION:")
print(f"  Sentiment : {sentiment}")
print(f"  Rating    : {rating}/5 stars")
print(f"  Raw output: '{output}'")